In [1]:
# from https://colab.research.google.com/drive/1ctgygDRJhVGUJTQy8-bRZCl1WNcT8De6?usp=sharing#scrollTo=WtvQLpd2MCi5
# and https://github.com/lm-sys/FastChat/blob/main/fastchat/llm_judge/README.md
from datasets import load_dataset
import pandas as pd

/home/ubuntu/skewed-score/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
dataset = load_dataset("lmsys/mt_bench_human_judgments")

dfs = []
for split_name, split in dataset.items():
    df = split.to_pandas()
    df["grader"] = split_name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

def conv_length(conv):
    return sum(len(turn["content"]) for turn in conv)

# Remove ties for simplicity
df = df_all[df_all['winner'].isin(['model_a', 'model_b'])].copy()

print(f"Rows after filtering: {len(df)}")

# Processing
df["len_a"] = df["conversation_a"].apply(conv_length)
df["len_b"] = df["conversation_b"].apply(conv_length)
df["length_diff"] = df["len_a"] - df["len_b"]
df["length_diff_prop"] = (df["len_a"] - df["len_b"]) / (df["len_a"] + df["len_b"])

# Create llm_pair (alphabetically sorted for canonical comparison)
df['llm_pair'] = df.apply(
    lambda row: '_vs_'.join(sorted([row['model_a'], row['model_b']])),
    axis=1
)

# Add individual model columns
df['left_model'] = df['model_a']   # Model in position A (left)
df['right_model'] = df['model_b']  # Model in position B (right)

# Create position variable (which model is on left/A position)
def get_position(row):
    """Returns 'left' if first alphabetical model is in position A, else 'right'"""
    models_sorted = sorted([row['model_a'], row['model_b']])
    return 'left' if row['model_a'] == models_sorted[0] else 'right'

df['position'] = df.apply(get_position, axis=1)

# Numeric position for modeling (-0.5 = left, +0.5 = right)
df['position_numeric'] = df['position'].map({'left': -0.5, 'right': 0.5})

# Outcome: was left side (model_a) chosen?
df["left_chosen"] = (df["winner"] == "model_a").astype(int)

# Drop dict columns
df = df.drop(columns=["conversation_a", "conversation_b"])

# Verify data structure
print("\n=== Data Structure Check ===")
print(f"Unique models: {df['left_model'].nunique()} (also {df['right_model'].nunique()})")
print(f"Models: {sorted(df['left_model'].unique())}")
print(f"\nUnique llm pairs: {df['llm_pair'].nunique()}")
print(f"\nUnique questions: {df['question_id'].nunique()}")
print(f"\nPosition distribution:")
print(df['position'].value_counts())
print(f"\nPosition numeric distribution:")
print(df['position_numeric'].value_counts())

# Verify counterbalancing
print(f"\nLeft chosen distribution:")
print(df['left_chosen'].value_counts())
print(f"Left chosen rate: {df['left_chosen'].mean():.3f}")

# Save to jsonl
df.to_json("data/all.jsonl", orient="records", lines=True)

print(f"\nSaved {len(df)} rows to data/all.jsonl")

Rows after filtering: 4375

=== Data Structure Check ===
Unique models: 6 (also 6)
Models: ['alpaca-13b', 'claude-v1', 'gpt-3.5-turbo', 'gpt-4', 'llama-13b', 'vicuna-13b-v1.2']

Unique llm pairs: 15

Unique questions: 80

Position distribution:
position
right    2424
left     1951
Name: count, dtype: int64

Position numeric distribution:
position_numeric
 0.5    2424
-0.5    1951
Name: count, dtype: int64

Left chosen distribution:
left_chosen
0    2892
1    1483
Name: count, dtype: int64
Left chosen rate: 0.339

Saved 4375 rows to data/all.jsonl


In [39]:
print(df['grader'].value_counts())

grader
human        2575
gpt4_pair    1800
Name: count, dtype: int64


In [40]:
print("="*60)
print("MODEL RANKINGS COMPARISON")
print("="*60)

# Overall win rate for each model (averaging both as left and right)
model_performance = {}

for model in df['left_model'].unique():
    # When this model is left_model
    as_left = df[df['left_model'] == model]['left_chosen'].mean()
    n_left = len(df[df['left_model'] == model])
    
    # When this model is right_model (chosen = NOT left_chosen)
    as_right = (1 - df[df['right_model'] == model]['left_chosen']).mean()
    n_right = len(df[df['right_model'] == model])
    
    # Average win rate
    overall_win_rate = (as_left * n_left + as_right * n_right) / (n_left + n_right)
    
    model_performance[model] = {
        'win_rate': overall_win_rate,
        'n_comparisons': n_left + n_right
    }

# Sort by win rate
ranking = sorted(model_performance.items(), key=lambda x: x[1]['win_rate'], reverse=True)

print("\nYour Data - Model Rankings:")
print(f"{'Rank':<6} {'Model':<25} {'Win Rate':<12} {'N Comparisons':<15}")
print("-" * 60)
for i, (model, stats) in enumerate(ranking, 1):
    print(f"{i:<6} {model:<25} {stats['win_rate']:.3f} ({stats['win_rate']:.1%})  {stats['n_comparisons']:<15}")

# Also break down by grader
print("\n" + "="*60)
print("By Grader:")
print("="*60)

for grader in ['gpt4_pair', 'human']:
    subset = df[df['grader'] == grader]
    
    model_perf_grader = {}
    for model in subset['left_model'].unique():
        as_left = subset[subset['left_model'] == model]['left_chosen'].mean()
        n_left = len(subset[subset['left_model'] == model])
        
        as_right = (1 - subset[subset['right_model'] == model]['left_chosen']).mean()
        n_right = len(subset[subset['right_model'] == model])
        
        if n_left + n_right > 0:
            overall = (as_left * n_left + as_right * n_right) / (n_left + n_right)
            model_perf_grader[model] = overall
    
    ranking_grader = sorted(model_perf_grader.items(), key=lambda x: x[1], reverse=True)
    
    print(f"\n{grader}:")
    for i, (model, win_rate) in enumerate(ranking_grader, 1):
        print(f"  {i}. {model:<25} {win_rate:.3f}")

MODEL RANKINGS COMPARISON

Your Data - Model Rankings:
Rank   Model                     Win Rate     N Comparisons  
------------------------------------------------------------
1      gpt-4                     0.858 (85.8%)  1469           
2      claude-v1                 0.738 (73.8%)  1321           
3      gpt-3.5-turbo             0.689 (68.9%)  1651           
4      vicuna-13b-v1.2           0.459 (45.9%)  1364           
5      alpaca-13b                0.200 (20.0%)  1407           
6      llama-13b                 0.062 (6.2%)  1538           

By Grader:

gpt4_pair:
  1. gpt-4                     0.932
  2. gpt-3.5-turbo             0.653
  3. alpaca-13b                0.168
  4. llama-13b                 nan
  5. vicuna-13b-v1.2           0.438

human:
  1. gpt-4                     0.797
  2. gpt-3.5-turbo             0.708
  3. claude-v1                 0.696
  4. vicuna-13b-v1.2           0.473
  5. alpaca-13b                0.225
  6. llama-13b                 0.084
